In [7]:
import pandas as pd
import os
import requests
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from transformers import DistilBertTokenizer, TFDistilBertModel
from PIL import Image
from io import BytesIO
import gc

In [8]:
# Load the datasets
df_train = pd.read_csv('dataset/train.csv')
df_test = pd.read_csv('dataset/test.csv')


class Config:
    KFOLDS = 5
    IMG_SIZE = 224
    MAX_TEXT_LEN = 128
    TEXT_MODEL_NAME = 'distilbert-base-uncased'
    TEXT_MODEL = TFDistilBertModel

    BATCH_SIZE = 32
    EPOCHS = 5
    LEARNING_RATE = 1e-4
    
    IMAGE_PREPROCESS_MODEL = keras.applications.efficientnet
    IMAGE_MODEL = keras.applications.EfficientNetB4
    IMAGE_MODEL_PARAMS = dict(
        include_top=False, 
        weights='imagenet', 
        pooling='avg' # Global Average Pooling
    )
    PREPROCESSED_DIR = 'preprocessed_images'

CONFIG = Config()

In [ ]:
def get_image_array(sample_id, url):
    filepath = os.path.join(CONFIG.PREPROCESSED_DIR, f"{sample_id}.npy")
    if os.path.exists(filepath):
        return np.load(filepath)
    
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content)).convert("RGB")
        img = img.resize((CONFIG.IMG_SIZE, CONFIG.IMG_SIZE))
        img_array = keras.preprocessing.image.img_to_array(img)
        processed_img = CONFIG.IMAGE_PREPROCESS_MODEL.preprocess_input(img_array)
        np.save(filepath, processed_img)
        return processed_img
    except (requests.exceptions.RequestException, IOError, ValueError) as e:
        print(f"Error with image {sample_id} ({url}): {e}. Using blank image.")
        return np.zeros((CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, 3), dtype=np.float32)

In [12]:
os.makedirs(CONFIG.PREPROCESSED_DIR, exist_ok=True)
tokenizer = DistilBertTokenizer.from_pretrained(CONFIG.TEXT_MODEL_NAME)

def preprocess_text(texts, tokenizer):
    return tokenizer(
        texts,
        max_length = CONFIG.MAX_TEXT_LEN,
        truncation = True,
        padding = 'max_length',
        return_tensors = 'tf'
    )

In [ ]:
print("Preprocessing text data...")
X_text_train = preprocess_text(df_train['catalog_content'].tolist(), tokenizer)
X_text_test = preprocess_text(df_test['catalog_content'].tolist(), tokenizer)

Preprocessing text data...


In [ ]:
print("Preprocessing image data (will download if not cached)...")
X_img_train = np.array([
    get_image_array(row.sample_id, row.image_link)
    for _, row in df_train.iterrows()
])
X_img_test = np.array([
    get_image_array(row.sample_id, row.image_link)
    for _, row in df_test.iterrows()
])

y_train = df_train['price'].values

In [ ]:
np.savez("dataset/data.npz",
        X_text_train = X_text_train,
        X_text_test = X_text_test,
        X_img_train = X_img_train,
        X_img_test = X_img_test,
        y_train = y_train
        )

In [ ]:
def create_multimodal_model():
    input_ids = keras.layers.Input(shape=(CONFIG.MAX_TEXT_LEN,), dtype=tf.int32, name='input_ids')
    attention_mask = keras.layers.Input(shape=(CONFIG.MAX_TEXT_LEN,), dtype=tf.int32, name='attention_mask')
    
    text_encoder = CONFIG.TEXT_MODEL.from_pretrained(CONFIG.TEXT_MODEL_NAME, trainable=True)
    text_embeddings = text_encoder(input_ids, attention_mask=attention_mask)[0]
    cls_token = text_embeddings[:, 0, :]
    text_features = keras.layers.Dense(128, activation='relu')(cls_token)

    image_input = keras.layers.Input(shape=(CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, 3), name='image_input')
    image_encoder = CONFIG.IMAGE_MODEL(**CONFIG.IMAGE_MODEL_PARAMS)
    image_encoder.trainable = True
    image_features = image_encoder(image_input)
    image_features = keras.layers.Dense(128, activation='relu')(image_features)

    concatenated = keras.layers.Concatenate()([text_features, image_features])
    x = keras.layers.Dropout(0.5)(concatenated)
    x = keras.layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    output = keras.layers.Dense(1, activation='relu', name='price')(x)
    
    model_inputs = {'input_ids': input_ids, 'attention_mask': attention_mask, 'image_input': image_input}
    return keras.Model(inputs=model_inputs, outputs=output)

In [ ]:
def smape_metric(y_true, y_pred):
    """Symmetric Mean Absolute Percentage Error for evaluation."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    numerator = tf.abs(y_pred - y_true)
    denominator = (tf.abs(y_true) + tf.abs(y_pred)) / 2
    # Add a small epsilon to avoid division by zero
    return tf.reduce_mean(numerator / (denominator + keras.backend.epsilon())) * 100

In [ ]:
df_train['price_bin'] = pd.cut(df_train['price'], bins=10, labels=False)
skf = StratifiedKFold(n_splits=CONFIG.KFOLDS, shuffle=True, random_state=42)

oof_predictions = np.zeros(len(df_train))
test_predictions = np.zeros(len(df_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(df_train, df_train['price_bin'])):
    print(f"\n===== FOLD {fold+1} / {CONFIG.KFOLDS} =====")
    
    X_train_fold = {
        'input_ids': X_text_train['input_ids'][train_idx],
        'attention_mask': X_text_train['attention_mask'][train_idx],
        'image_input': X_img_train[train_idx]
    }
    y_train_fold = y_train[train_idx]
    
    X_val_fold = {
        'input_ids': X_text_train['input_ids'][val_idx],
        'attention_mask': X_text_train['attention_mask'][val_idx],
        'image_input': X_img_train[val_idx]
    }
    y_val_fold = y_train[val_idx]
    
    X_test_fold = {
        'input_ids': X_text_test['input_ids'],
        'attention_mask': X_text_test['attention_mask'],
        'image_input': X_img_test
    }

    keras.backend.clear_session()
    model = create_multimodal_model()
    optimizer = keras.optimizers.Adam(learning_rate=CONFIG.LEARNING_RATE)
    model.compile(optimizer=optimizer, loss='mean_absolute_error', metrics=[smape_metric])
    
    model.fit(
        X_train_fold, y_train_fold,
        validation_data=(X_val_fold, y_val_fold),
        epochs=CONFIG.EPOCHS,
        batch_size=CONFIG.BATCH_SIZE,
        verbose=1
    )
    
    oof_preds_fold = model.predict(X_val_fold).flatten()
    oof_predictions[val_idx] = oof_preds_fold
    
    test_preds_fold = model.predict(X_test_fold).flatten()
    test_predictions += test_preds_fold / CONFIG.KFOLDS
    
    del model
    gc.collect()

In [ ]:
from sklearn.model_selection import StratifiedKFold
import tqdm

skf = StratifiedKFold(n_splits = 3, shuffle = True, random_state = 42)

for fold, (train, val) in enumerate(tqdm(skf.split(X, y), total = skf.get_n_splits(), desc = "Cross-validation"), start = 1) :
    print(f"--- Fold {fold} ---")
    X_train = X[train]
    y_train = y[train]
    X_val = X[val]
    y_val = y[val]

    print("\n--- Starting Model Training ---")
    history = model.fit(
        x = X_train,
        y = y_train,
        validation_data = (X_val, y_val),
        batch_size = CONFIG.BATCH_SIZE,
        epochs=CONFIG.EPOCHS
    )
    print("--- Model Training Finished ---")

    print("\n--- Generating Predictions on Test Set ---")
    predictions = model.predict(X_val)
    predicted_prices = predictions.flatten()

    